In [1]:
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
import re
import time

In [2]:
# Find project root automatically
# Expected structure:
#
# ROOT/
# ├── .github/workflows/
# ├── 42/dataset/workflow_security_labels.csv
# └── scripts/download_workflows.ipynb

current = Path.cwd().resolve()

ROOT_DIR = None

for candidate in [current, *current.parents]:
    labels_path = (
        candidate
        / "42"
        / "dataset"
        / "workflow_security_labels.csv"
    )

    if labels_path.exists():
        ROOT_DIR = candidate
        break

if ROOT_DIR is None:
    raise FileNotFoundError(
        "Could not find 42/dataset/workflow_security_labels.csv"
    )


DATASET_DIR = ROOT_DIR / "42" / "dataset"

LABELS_CSV = (
    DATASET_DIR
    / "workflow_security_labels.csv"
)

URLS_CSV = (
    DATASET_DIR
    / "workflow_urls.csv"
)

OUTPUT_DIR = (
    ROOT_DIR
    / ".github"
    / "workflows"
)

FAILURES_CSV = (
    DATASET_DIR
    / "workflow_download_failures.csv"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("ROOT_DIR   :", ROOT_DIR)
print("LABELS_CSV :", LABELS_CSV)
print("URLS_CSV   :", URLS_CSV)
print("OUTPUT_DIR :", OUTPUT_DIR)

ROOT_DIR   : /home/huawei/BOOSTSECURITY/42
LABELS_CSV : /home/huawei/BOOSTSECURITY/42/42/dataset/workflow_security_labels.csv
URLS_CSV   : /home/huawei/BOOSTSECURITY/42/42/dataset/workflow_urls.csv
OUTPUT_DIR : /home/huawei/BOOSTSECURITY/42/.github/workflows


In [3]:
df = pd.read_csv(LABELS_CSV)

if "workflow_blob_url" not in df.columns:
    raise ValueError(
        "Column 'workflow_blob_url' not found."
    )


workflow_urls = (
    df[["workflow_blob_url"]]
    .dropna()
    .copy()
)


workflow_urls["workflow_blob_url"] = (
    workflow_urls["workflow_blob_url"]
    .astype(str)
    .str.strip()
)


workflow_urls = workflow_urls[
    workflow_urls["workflow_blob_url"] != ""
]


workflow_urls = (
    workflow_urls
    .drop_duplicates()
    .reset_index(drop=True)
)


workflow_urls.to_csv(
    URLS_CSV,
    index=False
)


print("Total label rows :", len(df))
print("Unique workflows :", len(workflow_urls))
print("Saved to         :", URLS_CSV)

workflow_urls.head()

Total label rows : 2431
Unique workflows : 474
Saved to         : /home/huawei/BOOSTSECURITY/42/42/dataset/workflow_urls.csv


,workflow_blob_url
0,https://github.com/microsoft/ignite25-LAB506-m...
1,https://github.com/cloudposse-terraform-compon...
2,https://github.com/run-llama/liteparse/blob/a6...
3,https://github.com/fortinet-fortisoar/connecto...
4,https://github.com/fabric8io/kubernetes-client...


In [4]:
def github_to_raw_url(url):
    """
    Convert:

    https://github.com/owner/repo/blob/SHA/.github/workflows/test.yml

    to:

    https://raw.githubusercontent.com/owner/repo/SHA/.github/workflows/test.yml
    """

    parsed = urlparse(url)

    if parsed.netloc == "raw.githubusercontent.com":
        return url


    parts = parsed.path.strip("/").split("/")


    if (
        parsed.netloc not in {
            "github.com",
            "www.github.com"
        }
        or len(parts) < 5
        or parts[2] != "blob"
    ):
        raise ValueError(
            f"Invalid GitHub workflow URL: {url}"
        )


    owner = parts[0]
    repo = parts[1]
    ref = parts[3]

    file_path = "/".join(
        parts[4:]
    )


    return (
        f"https://raw.githubusercontent.com/"
        f"{owner}/{repo}/{ref}/{file_path}"
    )

In [5]:
def sanitize_filename(value):
    return re.sub(
        r"[^A-Za-z0-9._-]+",
        "-",
        str(value)
    ).strip("-")


def make_filename(url):
    """
    Example:

    https://github.com/cloudposse/test-helpers/blob/SHA/.github/workflows/test.yml

    becomes:

    cloudposse-test-helpers-test.yml
    """

    parsed = urlparse(url)

    parts = parsed.path.strip("/").split("/")

    if len(parts) < 5 or parts[2] != "blob":
        raise ValueError(
            f"Invalid GitHub workflow URL: {url}"
        )

    owner = sanitize_filename(parts[0])
    repo = sanitize_filename(parts[1])

    workflow_path = "/".join(parts[4:])

    original_filename = Path(
        workflow_path
    ).name

    original_filename = sanitize_filename(
        original_filename
    )

    return (
        f"{owner}-"
        f"{repo}-"
        f"{original_filename}"
    )

In [6]:
used_filenames = set()

filenames = []


for url in workflow_urls["workflow_blob_url"]:

    filename = make_filename(url)


    # If the same filename occurs more than once,
    # add part of the commit SHA
    if filename in used_filenames:

        parsed = urlparse(url)

        parts = parsed.path.strip("/").split("/")

        ref = parts[3]

        path = Path(filename)

        filename = (
            f"{path.stem}-"
            f"{ref[:8]}"
            f"{path.suffix}"
        )


        # Extra protection against another collision
        counter = 2

        while filename in used_filenames:

            filename = (
                f"{path.stem}-"
                f"{ref[:8]}-"
                f"{counter}"
                f"{path.suffix}"
            )

            counter += 1


    used_filenames.add(
        filename
    )

    filenames.append(
        filename
    )


workflow_urls = workflow_urls.copy()

workflow_urls[
    "filename"
] = filenames


print(
    "Unique workflow URLs:",
    len(workflow_urls)
)

print(
    "Unique filenames:",
    workflow_urls[
        "filename"
    ].nunique()
)


workflow_urls.head()

Unique workflow URLs: 474
Unique filenames: 474


,workflow_blob_url,filename
0,https://github.com/microsoft/ignite25-LAB506-m...,microsoft-ignite25-LAB506-migrate-and-moderniz...
1,https://github.com/cloudposse-terraform-compon...,cloudposse-terraform-components-aws-dms-iam-au...
2,https://github.com/run-llama/liteparse/blob/a6...,run-llama-liteparse-ocr_servers.yml
3,https://github.com/fortinet-fortisoar/connecto...,fortinet-fortisoar-connector-netscaler-adc-pyt...
4,https://github.com/fabric8io/kubernetes-client...,fabric8io-kubernetes-client-build.yml


In [7]:
MAX_RETRIES = 4

downloaded = 0
skipped = 0
failed = 0

failures = []


session = requests.Session()

session.headers.update(
    {
        "User-Agent":
        "github-actions-workflow-downloader"
    }
)


total = len(
    workflow_urls
)


for i, row in workflow_urls.iterrows():

    number = i + 1

    url = row[
        "workflow_blob_url"
    ]

    filename = row[
        "filename"
    ]

    output_path = (
        OUTPUT_DIR
        / filename
    )


    # Already downloaded
    if output_path.exists():

        skipped += 1

        print(
            f"[{number}/{total}] "
            f"SKIP   {filename}"
        )

        continue


    try:

        raw_url = github_to_raw_url(
            url
        )

    except Exception as e:

        failed += 1

        failures.append(
            {
                "workflow_blob_url": url,
                "filename": filename,
                "error": str(e)
            }
        )

        print(
            f"[{number}/{total}] "
            f"FAILED {filename}: {e}"
        )

        continue


    success = False
    last_error = None


    for attempt in range(
        1,
        MAX_RETRIES + 1
    ):

        try:

            response = session.get(
                raw_url,
                timeout=(10, 60)
            )


            response.raise_for_status()


            OUTPUT_DIR.mkdir(
                parents=True,
                exist_ok=True
            )


            output_path.write_bytes(
                response.content
            )


            downloaded += 1
            success = True


            print(
                f"[{number}/{total}] "
                f"OK     {filename}"
            )

            break


        except requests.HTTPError as e:

            last_error = e


            # Do not retry 404
            if (
                e.response is not None
                and
                e.response.status_code == 404
            ):
                break


        except (
            requests.ConnectionError,
            requests.Timeout
        ) as e:

            last_error = e


        except Exception as e:

            last_error = e

            break


        if attempt < MAX_RETRIES:

            wait = attempt * 2

            print(
                f"[{number}/{total}] "
                f"RETRY  {filename}"
            )

            time.sleep(
                wait
            )


    if not success:

        failed += 1

        failures.append(
            {
                "workflow_blob_url": url,
                "raw_url": raw_url,
                "filename": filename,
                "error": str(last_error)
            }
        )


        print(
            f"[{number}/{total}] "
            f"FAILED {filename}: "
            f"{last_error}"
        )

[1/474] OK     microsoft-ignite25-LAB506-migrate-and-modernize-windows-and-sql-server-workloads-to-azure-add-to-project.yml
[2/474] OK     cloudposse-terraform-components-aws-dms-iam-automerge.yml
[3/474] OK     run-llama-liteparse-ocr_servers.yml
[4/474] OK     fortinet-fortisoar-connector-netscaler-adc-pytest_checks.yml
[5/474] OK     fabric8io-kubernetes-client-build.yml
[6/474] OK     es-shims-all-node-tens.yml
[7/474] OK     aziontech-lib-code-review.yml
[8/474] OK     adafruit-Adafruit_CircuitPython_SCD30-failure-help-text.yml
[9/474] OK     Tencent-TencentKona-21-build-cross-compile.yml
[10/474] OK     cloudposse-terraform-components-aws-dns-primary-release.yml
[11/474] OK     turbot-steampipe-plugin-crtsh-golangci-lint.yml
[12/474] OK     salesforcecli-plugin-template-sf-test.yml
[13/474] OK     canonical-kepler-rocks-pull_request.yaml
[14/474] OK     AdobeDocs-audience-manager.ja-JP-assign-issue.yml
[15/474] OK     Hapag-Lloyd-.github-stale.yml
[16/474] OK     nextcloud-event_

In [8]:
print()
print("Download summary")
print("----------------")

print(
    "Unique workflows :",
    len(workflow_urls)
)

print(
    "Downloaded       :",
    downloaded
)

print(
    "Skipped existing :",
    skipped
)

print(
    "Failed           :",
    failed
)

print()
print(
    "Output directory :",
    OUTPUT_DIR
)


Download summary
----------------
Unique workflows : 474
Downloaded       : 472
Skipped existing : 0
Failed           : 2

Output directory : /home/huawei/BOOSTSECURITY/42/.github/workflows


In [9]:
failures_df = pd.DataFrame(
    failures
)


if not failures_df.empty:

    failures_df.to_csv(
        FAILURES_CSV,
        index=False
    )

    print(
        "Failures saved to:",
        FAILURES_CSV
    )

else:

    print(
        "No download failures."
    )


failures_df

Failures saved to: /home/huawei/BOOSTSECURITY/42/42/dataset/workflow_download_failures.csv


,workflow_blob_url,raw_url,filename,error
0,https://github.com/ministryofjustice/hmpps-per...,https://raw.githubusercontent.com/ministryofju...,ministryofjustice-hmpps-person-on-probation-us...,404 Client Error: Not Found for url: https://r...
1,https://github.com/Azure/terraform-azure-avm-u...,https://raw.githubusercontent.com/Azure/terraf...,Azure-terraform-azure-avm-utl-compute-windowsv...,404 Client Error: Not Found for url: https://r...


In [10]:
downloaded_files = sorted(
    list(
        OUTPUT_DIR.glob("*.yml")
    )
    +
    list(
        OUTPUT_DIR.glob("*.yaml")
    )
)


print(
    "Workflows currently in "
    ".github/workflows:",
    len(downloaded_files)
)

Workflows currently in .github/workflows: 472
